# Multi-LogiEval Analysis: CoT vs Lean vs Two-Stage

Comprehensive analysis comparing three approaches across all depths (d1-d5) and logic types (FOL, NM, PL).

**Dataset**: 150 questions (3 logic types × 5 depths × 10 samples)
- **FOL**: First-order logic
- **NM**: Non-monotonic logic
- **PL**: Propositional logic
- **Depths**: d1 (1 rule) through d5 (5 chained rules)

**Research Question**: Does Lean verification help or hurt on Multi-LogiEval compared to FOLIO?

In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
import numpy as np

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)

## 1. Load Results

In [ ]:
# Load all three result sets
import os
project_root = os.path.abspath(os.path.join(os.getcwd(), '../..' if 'nb/' in os.getcwd() else '.'))

with open(os.path.join(project_root, 'results/multilogieval/all_depths/cot/all_results.json'), 'r') as f:
    cot_results = json.load(f)

with open(os.path.join(project_root, 'results/multilogieval/all_depths/lean/all_results.json'), 'r') as f:
    lean_results = json.load(f)

with open(os.path.join(project_root, 'results/multilogieval/all_depths/two_stage/all_results.json'), 'r') as f:
    two_stage_results = json.load(f)

print(f"Loaded {len(cot_results)} questions from each approach")
print(f"\nOverall Accuracies:")
print(f"  CoT:       {sum(1 for r in cot_results if r['correct'])}/150 = {sum(1 for r in cot_results if r['correct'])/150*100:.2f}%")
print(f"  Lean:      {sum(1 for r in lean_results if r['correct'])}/150 = {sum(1 for r in lean_results if r['correct'])/150*100:.2f}%")
print(f"  Two-Stage: {sum(1 for r in two_stage_results if r['correct'])}/150 = {sum(1 for r in two_stage_results if r['correct'])/150*100:.2f}%")

## 2. Overall Performance Comparison

In [ ]:
cot_acc = sum(1 for r in cot_results if r['correct']) / len(cot_results) * 100
lean_acc = sum(1 for r in lean_results if r['correct']) / len(lean_results) * 100
two_stage_acc = sum(1 for r in two_stage_results if r['correct']) / len(two_stage_results) * 100

# Visualize
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

approaches = ['CoT', 'Lean', 'Two-Stage']
accuracies = [cot_acc, lean_acc, two_stage_acc]
colors = ['#2ecc71', '#3498db', '#9b59b6']

bars = ax1.bar(approaches, accuracies, color=colors)
ax1.set_ylabel('Accuracy (%)', fontsize=12)
ax1.set_title('Overall Accuracy Comparison', fontsize=14, fontweight='bold')
ax1.set_ylim(0, 100)
for bar, acc in zip(bars, accuracies):
    height = bar.get_height()
    ax1.text(bar.get_x() + bar.get_width()/2., height + 2,
            f'{acc:.1f}%', ha='center', fontsize=11, fontweight='bold')

# Correct/Incorrect breakdown
cot_correct = sum(1 for r in cot_results if r['correct'])
lean_correct = sum(1 for r in lean_results if r['correct'])
two_stage_correct = sum(1 for r in two_stage_results if r['correct'])

x = np.arange(3)
correct_counts = [cot_correct, lean_correct, two_stage_correct]
incorrect_counts = [150 - c for c in correct_counts]

ax2.bar(x, correct_counts, label='Correct', color='#2ecc71')
ax2.bar(x, incorrect_counts, bottom=correct_counts, label='Incorrect', color='#e74c3c')
ax2.set_xticks(x)
ax2.set_xticklabels(approaches)
ax2.set_ylabel('Number of Questions', fontsize=12)
ax2.set_title('Correct vs Incorrect Breakdown', fontsize=14, fontweight='bold')
ax2.legend()

plt.tight_layout()
plt.show()

print("\n" + "="*70)
print("COMPARISON WITH FOLIO")
print("="*70)
print(f"FOLIO:          CoT (85.7%) > Two-Stage (79.3%) > Lean (74.9%)")
print(f"Multi-LogiEval: CoT ({cot_acc:.1f}%) vs Lean ({lean_acc:.1f}%) vs Two-Stage ({two_stage_acc:.1f}%)")
print("="*70)

## 3. Lean Verification Statistics

In [ ]:
# Get verified-but-wrong cases
lean_verified = sum(1 for r in lean_results if r.get('lean_verification') and r.get('lean_verification').get('success', False))
lean_verified_correct = sum(1 for r in lean_results 
                            if r.get('lean_verification') and r.get('lean_verification').get('success', False) and r['correct'])
lean_verified_wrong = sum(1 for r in lean_results 
                          if r.get('lean_verification') and r.get('lean_verification').get('success', False) and not r['correct'])

two_stage_verified = 0
two_stage_verified_correct = 0
two_stage_verified_wrong = 0
for r in two_stage_results:
    if 'all_cycles' in r and r['all_cycles']:
        final_cycle = r['all_cycles'][-1]
        if final_cycle.get('lean_success', False):
            two_stage_verified += 1
            if r['correct']:
                two_stage_verified_correct += 1
            else:
                two_stage_verified_wrong += 1

print("="*70)
print("LEAN VERIFICATION STATISTICS")
print("="*70)

print(f"\nLean:")
print(f"  Verification success: {lean_verified}/150 ({lean_verified/150*100:.1f}%)")
if lean_verified > 0:
    print(f"  Among verified:")
    print(f"    Correct answer:   {lean_verified_correct} ({lean_verified_correct/lean_verified*100:.1f}%)")
    print(f"    Wrong answer:     {lean_verified_wrong} ({lean_verified_wrong/lean_verified*100:.1f}%)")

print(f"\nTwo-Stage:")
print(f"  Verification success: {two_stage_verified}/150 ({two_stage_verified/150*100:.1f}%)")
if two_stage_verified > 0:
    print(f"  Among verified:")
    print(f"    Correct answer:   {two_stage_verified_correct} ({two_stage_verified_correct/two_stage_verified*100:.1f}%)")
    print(f"    Wrong answer:     {two_stage_verified_wrong} ({two_stage_verified_wrong/two_stage_verified*100:.1f}%)")

print(f"\nKEY INSIGHT:")
if lean_verified > 0:
    print(f"Lean has {lean_verified_wrong} verified-but-wrong cases ({lean_verified_wrong/lean_verified*100:.1f}%)")
if two_stage_verified > 0:
    print(f"Two-Stage has {two_stage_verified_wrong} verified-but-wrong cases ({two_stage_verified_wrong/two_stage_verified*100:.1f}%)")
print("="*70)

## 4. Performance by Logic Type

In [ ]:
logic_types = ['fol', 'nm', 'pl']
logic_names = {'fol': 'FOL', 'nm': 'NM', 'pl': 'PL'}

cot_by_logic = {}
lean_by_logic = {}
two_stage_by_logic = {}

for logic in logic_types:
    cot_logic = [r for r in cot_results if r['logic_type'] == logic]
    lean_logic = [r for r in lean_results if r['logic_type'] == logic]
    two_stage_logic = [r for r in two_stage_results if r['logic_type'] == logic]
    
    cot_by_logic[logic] = sum(1 for r in cot_logic if r['correct']) / len(cot_logic) * 100
    lean_by_logic[logic] = sum(1 for r in lean_logic if r['correct']) / len(lean_logic) * 100
    two_stage_by_logic[logic] = sum(1 for r in two_stage_logic if r['correct']) / len(two_stage_logic) * 100

# Visualize
fig, ax = plt.subplots(figsize=(12, 6))

x = np.arange(len(logic_types))
width = 0.25

ax.bar(x - width, [cot_by_logic[l] for l in logic_types], width, label='CoT', color='#2ecc71')
ax.bar(x, [lean_by_logic[l] for l in logic_types], width, label='Lean', color='#3498db')
ax.bar(x + width, [two_stage_by_logic[l] for l in logic_types], width, label='Two-Stage', color='#9b59b6')

ax.set_xlabel('Logic Type', fontsize=12)
ax.set_ylabel('Accuracy (%)', fontsize=12)
ax.set_title('Accuracy by Logic Type', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels([logic_names[l] for l in logic_types])
ax.legend()
ax.set_ylim(0, 100)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

print("\n" + "="*70)
print("ACCURACY BY LOGIC TYPE")
print("="*70)
print(f"{'Logic':<8} {'CoT':<12} {'Lean':<12} {'Two-Stage':<12}")
print("-"*70)
for logic in logic_types:
    print(f"{logic_names[logic]:<8} {cot_by_logic[logic]:>6.1f}%     {lean_by_logic[logic]:>6.1f}%     {two_stage_by_logic[logic]:>6.1f}%")
print("="*70)

## 5. Performance by Depth

In [ ]:
depths = ['d1_Data', 'd2_Data', 'd3_Data', 'd4_Data', 'd5_Data']
depth_labels = ['d1', 'd2', 'd3', 'd4', 'd5']

cot_by_depth = {}
lean_by_depth = {}
two_stage_by_depth = {}

for depth in depths:
    cot_depth = [r for r in cot_results if r['depth_dir'] == depth]
    lean_depth = [r for r in lean_results if r['depth_dir'] == depth]
    two_stage_depth = [r for r in two_stage_results if r['depth_dir'] == depth]
    
    cot_by_depth[depth] = sum(1 for r in cot_depth if r['correct']) / len(cot_depth) * 100
    lean_by_depth[depth] = sum(1 for r in lean_depth if r['correct']) / len(lean_depth) * 100
    two_stage_by_depth[depth] = sum(1 for r in two_stage_depth if r['correct']) / len(two_stage_depth) * 100

# Visualize
fig, ax = plt.subplots(figsize=(12, 6))

x = np.arange(len(depths))
ax.plot(x, [cot_by_depth[d] for d in depths], 'o-', label='CoT', color='#2ecc71', linewidth=2, markersize=8)
ax.plot(x, [lean_by_depth[d] for d in depths], 's-', label='Lean', color='#3498db', linewidth=2, markersize=8)
ax.plot(x, [two_stage_by_depth[d] for d in depths], '^-', label='Two-Stage', color='#9b59b6', linewidth=2, markersize=8)

ax.set_xlabel('Reasoning Depth (Number of Chained Rules)', fontsize=12)
ax.set_ylabel('Accuracy (%)', fontsize=12)
ax.set_title('Accuracy by Reasoning Depth', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(depth_labels)
ax.legend(fontsize=11)
ax.set_ylim(0, 100)
ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

print("\n" + "="*70)
print("ACCURACY BY DEPTH")
print("="*70)
print(f"{'Depth':<8} {'CoT':<12} {'Lean':<12} {'Two-Stage':<12}")
print("-"*70)
for depth, label in zip(depths, depth_labels):
    print(f"{label:<8} {cot_by_depth[depth]:>6.1f}%     {lean_by_depth[depth]:>6.1f}%     {two_stage_by_depth[depth]:>6.1f}%")
print("="*70)

## 6. Summary

In [ ]:
print("="*70)
print("SUMMARY: MULTI-LOGIEVAL RESULTS")
print("="*70)

print(f"\n1. OVERALL ACCURACY:")
print(f"   CoT:       {cot_acc:.1f}%")
print(f"   Lean:      {lean_acc:.1f}%")
print(f"   Two-Stage: {two_stage_acc:.1f}%")

print(f"\n2. VERIFIED-BUT-WRONG CASES:")
print(f"   Lean:      {lean_verified_wrong} cases ({lean_verified_wrong/lean_verified*100:.1f}% of verified)")
print(f"   Two-Stage: {two_stage_verified_wrong} cases ({two_stage_verified_wrong/two_stage_verified*100:.1f}% of verified)")

print(f"\n3. NEXT STEPS:")
print(f"   - Analyze {lean_verified_wrong} Lean verified-but-wrong cases to find error patterns")
print(f"   - Analyze {two_stage_verified_wrong} Two-Stage verified-but-wrong cases")
print(f"   - Compare error types with FOLIO findings")
print("="*70)